# Stage 3 — DINOv2 fingertip-pool + temporal context + visibility gate (Kaggle T4)

Tests the strengthened fingertip-pool design after Stage 2 (mean-pool) failed at CER 0.86 and Stage 1 v3 falsified the DANN hypothesis.

**Recipe** (locked from Stage 1 v2; see `configs/stage3.yaml` for the contract):
- DINOv2-S/14 at **336x336** → 24x24 = 576 patches per frame.
- **Bell-weighted 3x3 patch window** around MediaPipe `INDEX_FINGER_TIP` (joint 8), edge-clamped.
- **±1 temporal context concat** → 3·D = 1152 channels.
- **Visibility gate**: zero the context block on MediaPipe dropouts and append a 0/1 vis bit → final per-frame feature is **1153-dim**.
- Pre-projection `LayerNorm(1153)` in the Conformer.  Optimiser / scheduler / augmentation / seed / Conformer config = Stage 1 v2.

**Pass/fail** (per plan §2 Task B):
- Every fold must reach **train NLL < 0.5** within 80 epochs — gating test.
- Headline (full cohort): mean val CER ≤ **0.860** (beat Stage 2).
- Headline (PHW/KIM-stripped per HRNet null verdict): mean val CER must improve on the Stage 1 v3 stripped baseline **0.6383 ± 0.0445**.
- Stretch (full): ≤ 0.78 (Stage 4 fusion likely clears 0.55).

**Wall-clock estimate on Kaggle T4 (single session)**:

| Phase | Time |
|---|---|
| Cache build (3477 clips × DINOv2-S/14 at 336x336 + MediaPipe per frame) | ~100 min |
| 5-fold sweep (5 × ~40 min, batch=32, 80 epochs) | ~3 h 30 min |
| Total | **~5 h** |

Kaggle's 9 h kernel limit accommodates this with margin, but the **recommended** workflow is **two kernels**:
1. Kernel 1: build the cache + manifest, commit so the cache becomes a notebook-output dataset (~2 h).
2. Kernel 2: attach Kernel 1's output, skip the build, run only the sweep (~3.5 h).  Cheaper to restart if anything goes sideways.

The notebook supports both workflows via Cell 2's resume logic — same cells, just attach whatever you have.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy transformers --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU            : {name}  ({mem:.1f} GB)')

## Cell 2 — Locate artifacts + config (Stage 3 hyperparams)

Auto-finds the Stage 3 cache, CV manifest, and incremental results under `/kaggle/working/` or `/kaggle/input/**` (any attached notebook-output dataset).

Outputs to `/kaggle/working/` so they survive the session via kernel commit.

In [ ]:
import os, logging, random, json, glob
import numpy as np
import torch

def _find(pattern: str):
    matches = (
        glob.glob(f'/kaggle/working/**/{pattern}', recursive=True)
        + glob.glob(f'/kaggle/input/**/{pattern}',  recursive=True)
    )
    return matches[0] if matches else None

STAGE3_CACHE = _find('stage3_features.pt')
CV_MANIFEST  = _find('subject_cv5.json')
RESULTS_PATH_FOUND = _find('stage3_results.json')

# Always WRITE to /kaggle/working/.  We read from wherever the find() located.
OUT_CACHE    = STAGE3_CACHE   or '/kaggle/working/stage3_features.pt'
OUT_MANIFEST = CV_MANIFEST    or '/kaggle/working/subject_cv5.json'
RESULTS_PATH = '/kaggle/working/stage3_results.json'
# If a results JSON exists in /kaggle/input/, copy it to /kaggle/working/ so
# we resume incrementally without clobbering the source.
if RESULTS_PATH_FOUND and not os.path.exists(RESULTS_PATH):
    import shutil; shutil.copy(RESULTS_PATH_FOUND, RESULTS_PATH)

CKPT_DIR = '/kaggle/working/checkpoints'
LOG_DIR  = '/kaggle/working/logs'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[logging.StreamHandler(),
              logging.FileHandler(os.path.join(LOG_DIR, 'stage3.log'))],
)

from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig

# Stage 3 locked hyperparams — DO NOT MODIFY per-fold.
T_NATIVE         = 32
IMAGE_SIZE       = 336
TEMPORAL_CONTEXT = True
VISIBILITY_GATE  = True
MULTI_JOINT      = False         # set True for the ablation
BELL_SIGMA       = 1.0
UPSAMPLE         = 2
D_MODEL          = 256
N_LAYERS         = 4
N_HEADS          = 4
CONV_KERNEL      = 15
DROPOUT          = 0.2
BATCH_SIZE       = 32
LR_PEAK          = 5e-4
WEIGHT_DECAY     = 5e-2
GRAD_CLIP        = 1.0
NUM_EPOCHS       = 80
WARMUP_PCT       = 0.05
SEED             = 42
FOLDS            = list(range(5))
VARIANT_NAME     = 'stage3_multijoint' if MULTI_JOINT else 'stage3'

# EncoderConfig.arch='siglip' is a benign placeholder — the trainer reads
# features straight from the Stage 3 cache and never touches cfg.encoder.
cfg = Config(
    data=DataConfig(
        hf_repo_id='yewon816/WiTA', lang='english',
        max_zips=None, max_frames=64,
        train_split=0.90, seed=SEED,
    ),
    encoder=EncoderConfig(arch='siglip'),
    train=TrainConfig(
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr=LR_PEAK,
        weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
        num_workers=2, warmup_pct=WARMUP_PCT, seed=SEED,
        checkpoint_dir=CKPT_DIR,
    ),
).build()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

print(f'Device         : {cfg.device}')
print(f'Stage 3 cache  : {OUT_CACHE}  (exists={os.path.exists(OUT_CACHE)})')
print(f'CV manifest    : {OUT_MANIFEST}  (exists={os.path.exists(OUT_MANIFEST)})')
print(f'Results path   : {RESULTS_PATH}  (exists={os.path.exists(RESULTS_PATH)})')
print(f'Variant        : {VARIANT_NAME}')
print(f'Image size     : {IMAGE_SIZE}x{IMAGE_SIZE}  (24x24 patch grid)')

## Cell 3 — Stream WiTA data (skip if Stage 3 cache already exists)

Streams the 39 English subset zips, indexes them, and holds them in memory for Cell 4 to consume.  ~15 min on Kaggle network.  Skipped entirely if Cell 2 located an existing cache.

In [ ]:
from wita_v2.datasets.subject_splits import stream_and_index_with_subjects
if os.path.exists(OUT_CACHE):
    print(f'Cache exists — skipping streaming.')
    samples = None
else:
    samples = stream_and_index_with_subjects(cfg)
    print(f'Total: {len(samples)} clips across {len({s for _,_,s in samples})} subjects')

## Cell 4 — Build (or load) the Stage 3 fingertip-pool cache

**Wall-clock**: ~100 min on T4 for 3477 clips (DINOv2 forward over 64-frame clips at 336x336 + per-frame MediaPipe).

If you hit the 9 h session limit during the sweep, commit this kernel after Cell 4 completes — the cache is the expensive artifact and shouldn't be rebuilt.

In [ ]:
from wita_v2.models.encoders.dinov2_encoder import DINOv2Encoder
from wita_v2.datasets.dinov2_feature_cache import (
    extract_dinov2_feature_cache, check_fingerprint_match,
)

if not os.path.exists(OUT_CACHE):
    encoder = DINOv2Encoder(
        model_name='facebook/dinov2-small',
        image_size=IMAGE_SIZE, pool='mean_patch',  # pool flag unused on patch path
    )
    extract_dinov2_feature_cache(
        samples=samples, encoder=encoder, out_path=OUT_CACHE,
        T_native=T_NATIVE,
        temporal_context=TEMPORAL_CONTEXT,
        visibility_gate=VISIBILITY_GATE,
        multi_joint=MULTI_JOINT,
        bell_sigma=BELL_SIGMA,
        device=cfg.device, dtype=torch.float16,
    )
    del encoder
    torch.cuda.empty_cache()
    print('cache built.')

cache = torch.load(OUT_CACHE, map_location='cpu', weights_only=False)
expected_fp = {
    'model_name':       'facebook/dinov2-small',
    'image_size':       IMAGE_SIZE,
    'grid_size':        IMAGE_SIZE // 14,
    'patch_size':       14,
    'pool':             'fingertip_3x3_bell',
    'temporal_context': TEMPORAL_CONTEXT,
    'visibility_gate':  VISIBILITY_GATE,
    'multi_joint':      MULTI_JOINT,
}
check_fingerprint_match(cache, expected_fp)
print(f'Loaded cache: {len(cache["feats"])} clips, '
      f'in_dim={cache["out_dim"]}, '
      f'detect_rate={cache.get("frame_detect_rate", 0)*100:.1f}%')

## Cell 5 — Build (or load) 5-fold subject CV manifest

Re-uses the same seed=42 manifest as Stage 1 v3 so fold splits are pairwise comparable across stages.

In [ ]:
from wita_v2.datasets.cv_splits import build_cv5_manifest, save_cv5_manifest, load_cv5_manifest

if os.path.exists(OUT_MANIFEST):
    manifest = load_cv5_manifest(OUT_MANIFEST)
    print(f'Loaded CV manifest from {OUT_MANIFEST}')
else:
    fake_samples = [(b'', cache['labels'][i], cache['subjects'][i])
                    for i in range(len(cache['feats']))]
    manifest = build_cv5_manifest(fake_samples, n_folds=5, seed=SEED)
    save_cv5_manifest(manifest, OUT_MANIFEST)

print(f'\nCV manifest: {manifest["n_subjects_total"]} signers, {manifest["n_folds"]} folds')
for f in manifest['folds']:
    print(f'  fold {f["fold"]}: train={f["n_train_subjects"]} ({f["n_train_clips"]} clips)  '
          f'val={f["n_val_subjects"]} ({f["n_val_clips"]} clips)')

## Cell 6 — Run the 5-fold sweep

Each fold uses `train_one_fold` from `training/stage3_train.py`.  Results are written incrementally so the notebook tolerates session disconnects.  Re-running this cell after a disconnect skips already-completed folds.

In [ ]:
from wita_v2.training.stage3_train  import train_one_fold
from wita_v2.datasets.cv_splits      import fold_indices
from wita_v2.datasets.skeleton_augment import LandmarkAugment

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
    completed = {(r['fold'], r['variant']) for r in all_results}
    print(f'Resuming — {len(completed)} folds already complete.')
else:
    all_results = []
    completed = set()

# Temporal-only augmentation — spatial jitter disabled per Stage 3 contract.
train_aug = LandmarkAugment(p_spatial_jitter=0.0, spatial_sigma=0.0)

subject_per_idx = cache['subjects']
for fold in FOLDS:
    if (fold, VARIANT_NAME) in completed:
        continue
    train_idx, val_idx = fold_indices(manifest, fold, subject_per_idx)
    result = train_one_fold(
        cache=cache, train_idx=train_idx, val_idx=val_idx, cfg=cfg,
        fold=fold, variant=VARIANT_NAME,
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr_peak=LR_PEAK,
        weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP, dropout=DROPOUT,
        d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
        conv_kernel=CONV_KERNEL, upsample=UPSAMPLE,
        warmup_pct=WARMUP_PCT, transform=train_aug,
        checkpoint_dir=CKPT_DIR, log_dir=LOG_DIR,
    )
    summary = {k: v for k, v in result.items() if k != 'history'}
    all_results.append(summary)
    with open(RESULTS_PATH, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'  saved → {RESULTS_PATH}  (total folds done: {len(all_results)})\n')

print(f'\nAll {len(all_results)}/{len(FOLDS)} folds done.')

## Cell 7 — Aggregate: per-fold table + 5-fold mean ± std (both cohorts)

In [ ]:
import numpy as np
from wita_v2.reports.template.stripped_cohort import dual_cohort_summary

with open(RESULTS_PATH) as f:
    all_results = json.load(f)

per_fold = {r['fold']: r for r in all_results if r['variant'] == VARIANT_NAME}
best_cers  = [per_fold[f]['best_val_cer']         for f in FOLDS if f in per_fold]
final_nlls = [per_fold[f]['final_train_nll']      for f in FOLDS if f in per_fold]
nll_below  = [per_fold[f]['train_nll_ever_below_05'] for f in FOLDS if f in per_fold]

print(f'Stage 3 ({VARIANT_NAME}) — {len(best_cers)} folds complete\n')
print(' fold    best_val_cer   final_train_nll    train_nll<0.5')
print(' ----    ------------   ---------------    -------------')
for f in FOLDS:
    if f not in per_fold:
        print(f'  {f:>2d}    {"-":>10}     {"-":>13}      {"-":>10}')
        continue
    r = per_fold[f]
    print(f'  {f:>2d}    {r["best_val_cer"]:.4f}        {r["final_train_nll"]:.4f}            '
          f'{"yes" if r["train_nll_ever_below_05"] else "NO"}')

if len(best_cers) >= 2:
    s = dual_cohort_summary(RESULTS_PATH, VARIANT_NAME)
    print(f'\n     full cohort   : {s["full_mean"]:.4f} ± {s["full_std"]:.4f}  (n={len(s["folds"])})')
    print(f'     PHW/KIM-strip : {s["stripped_mean"]:.4f} ± {s["stripped_std"]:.4f}')
    print(f'     Δ (full-strip): {s["delta_mean"]:+.4f}')

## Cell 8 — Verdict per Stage 3 pass/fail thresholds (dual-cohort)

Compares both the full and PHW/KIM-stripped means against the gating + headline thresholds.

In [ ]:
if len(best_cers) >= 2:
    full_mean    = s['full_mean']
    strip_mean   = s['stripped_mean']
    print('\n=== Stage 3 verdict ===')
    print(f'  variant            : {VARIANT_NAME}')
    print(f'  full cohort        : {full_mean:.4f} ± {s["full_std"]:.4f}')
    print(f'  PHW/KIM-stripped   : {strip_mean:.4f} ± {s["stripped_std"]:.4f}')
    print(f'  Δ (full-strip)     : {s["delta_mean"]:+.4f}')

    if not all(nll_below):
        n_fail = sum(1 for b in nll_below if not b)
        print(f'\n  ❌ GATING TEST FAILED: {n_fail}/{len(nll_below)} folds did NOT reach train NLL < 0.5.')
        print('     The fingertip-pool design is feature-insufficient.')
        print('     Investigate before running Stage 4 — fusion will not save it.')
    else:
        if full_mean <= 0.78:
            print(f'\n  ✅ FULL STRETCH ({full_mean:.4f} ≤ 0.78) — Stage 4 likely clears 0.55.')
        elif full_mean <= 0.860:
            print(f'\n  ✅ FULL HEADLINE ({full_mean:.4f} ≤ 0.860) — beats Stage 2 mean.')
        else:
            print(f'\n  ❌ FULL FAIL: mean {full_mean:.4f} > 0.860 — does not beat Stage 2.')
        STAGE1V3_STRIPPED = 0.6383
        if strip_mean < STAGE1V3_STRIPPED - 0.01:
            print(f'  ✅ STRIPPED WIN ({strip_mean:.4f} < {STAGE1V3_STRIPPED:.4f} - 0.01) — model-side progress.')
        elif strip_mean > STAGE1V3_STRIPPED + 0.01:
            print(f'  ❌ STRIPPED LOSS ({strip_mean:.4f} > {STAGE1V3_STRIPPED:.4f} + 0.01) — worse than Stage 1 v3 stripped baseline.')
        else:
            print(f'  ⚖  STRIPPED TIE ({strip_mean:.4f} ≈ {STAGE1V3_STRIPPED:.4f}) — Stage 4 fusion needed to break out.')

## Cell 9 — Per-signer val CER scatter

Shows whether Stage 3 closes the model-side hard regime (PJH, SYB, KJM, KNY, LKS, YMG) or whether the easy regime carries the gain.  PHW and KIM stay marked but are dataset-side limits (HRNet swap experiment, verdict null).

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

per_signer = {}
for r in all_results:
    if r['variant'] != VARIANT_NAME: continue
    per_signer.update(r.get('best_per_signer_val_cer', {}) or {})
items = sorted(per_signer.items(), key=lambda kv: kv[1])
xs = list(range(len(items)))
ys = [v for _, v in items]
DATASET_LIMIT = ['PHW', 'KIM']
MODEL_HARD    = ['PJH','SYB','KJM','KNY','LKS','YMG']
def _color(s):
    if s in DATASET_LIMIT: return '#7f7f7f'   # grey: dataset-side
    if s in MODEL_HARD:    return '#d62728'   # red: model-side hard
    return '#1f77b4'                          # blue: easy / mid
colors = [_color(s) for s,_ in items]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.scatter(xs, ys, s=30, c=colors)
ax.axhline(0.55, color='green', linestyle='--', alpha=0.5, label='easy ≤ 0.55')
ax.axhline(0.75, color='red',   linestyle='--', alpha=0.5, label='hard ≥ 0.75')
ax.set_xticks(xs)
ax.set_xticklabels([s for s,_ in items], rotation=90, fontsize=7)
ax.set_ylabel('val CER at best-epoch')
ax.set_xlabel('signer (sorted by CER)')
ax.set_title(f'Stage 3 ({VARIANT_NAME}) — per-signer val CER  '
             '[grey=PHW/KIM dataset-side, red=model-side hard]')
ax.legend(frameon=False)
ax.grid(True, linestyle=':', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'stage3_per_signer_scatter.png'), dpi=140)
plt.show()

## Cell 10 — Commit kernel

**Save & Run All (Commit)** so these artifacts survive the session and can be attached to Stage 4 / Stage 5 kernels:
- `/kaggle/working/stage3_features.pt`  — the cache (expensive to rebuild, ~100 min)
- `/kaggle/working/subject_cv5.json`    — CV manifest
- `/kaggle/working/stage3_results.json` — per-fold incremental results
- `/kaggle/working/checkpoints/stage3_fold*_best.pt` — 5 best checkpoints, one per fold
- `/kaggle/working/logs/stage3_per_signer_scatter.png` — Cell 9 figure for the report

Stage 4 (fusion) requires this cache attached + the Stage 1 v3 skeleton cache.